# Dataset Generation for Homoglyph Detection

This notebook provides the code for generating the dataset for homoglpyh detection

Dependencies:
- qahirah
- freetype
- `vis_gen.py` from `source/vis_gen.py`

## 1. Download Unicode Metadata files

### 1.1 Download UnicodeData.txt file
The file `UnicodeData.txt` from the Unicode Character Database (UCD) provides detailed metadata for every Unicode code point.

In [ ]:
import requests

url = "https://www.unicode.org/Public/UCD/latest/ucd/UnicodeData.txt"
response = requests.get(url)

# Save to file
with open("UnicodeData.txt", "w", encoding="utf-8") as f:
    f.write(response.text)

print("Downloaded UnicodeData.txt")

### 1.2 Download Blocks.txt file

The file `Blocks.txt` defines the ranges of code points assigned to named blocks in the Unicode standard (e.g., Basic Latin, Greek and Coptic, Arabic). Each line specifies a start and end code point along with the block name.

In [ ]:
import requests

url = "https://www.unicode.org/Public/UCD/latest/ucd/Blocks.txt"
response = requests.get(url)

# Save to file
with open("Blocks.txt", "w", encoding="utf-8") as f:
    f.write(response.text)

print("Downloaded Blocks.txt")

## 2. Gather required codepoints

### 2.1 Extract the left-to-right scripts

In [ ]:
alphabets = {}

with open('/content/Blocks.txt', 'r') as infile:
  for line in infile.readlines():
    # Skip lines that are comments or empty
    if line[0]=='#' or line[0]=='\n':
      continue

    # Each line has the format: <start>..<end>; <Block Name>
    parts = line.split(";")
    codepoints = parts[0]
    script = parts[1].strip()

    # Split the codepoint range into start and end hex values
    start_codepoint = codepoints.split('..')[0]
    end_codepoint = codepoints.split('..')[1]

    # Include only scripts related to alphabets by checking for specific substrings in the block name
    if any(key in script.lower() for key in ('latin', 'cyrillic', 'armenian',
                                             'greek', 'coptic', 'ipa',
                                             'spacing', 'diacritical','georgian',
                                             'hangul','ethiopic', 'cherokee',
                                             'canadian', 'ogham', 'runic',
                                             'tagalog','hanunoo','buhid',
                                             'tagbanwa')):
      # Exclude CJK (Chinese, Japanese, Korean)
      if 'cjk' not in script.lower():
        alphabets[script] = {
            'start': start_codepoint,
            'end': end_codepoint
        }

# Sort the dictionary
alphabets = dict(sorted(alphabets.items()))

In [ ]:
alphabets

### 2.2 Extract the codepoints

In [ ]:
import os

# Set of general categories for characters that are considered renderable
renderable_categories = {
    'Lu', 'Ll', 'Lt', 'Lm', 'Lo',  # Letters
    'Mn', 'Mc', 'Me',              # Marks
    'Nd', 'Nl', 'No',              # Numbers
    'Pc', 'Pd', 'Ps', 'Pe', 'Pi', 'Pf', 'Po',  # Punctuation
    'Sm', 'Sc', 'Sk', 'So',        # Symbols
    # 'Zs'                           # Space separator
}

# Directory to store output files for each script
scripts_dir = '/content/scripts'
os.makedirs(scripts_dir, exist_ok=True)

# Loop over each script and its corresponding codepoint range in the 'alphabets' dictionary
for script, range_info in alphabets.items():
  start_codepoint = range_info['start']
  end_codepoint = range_info['end']

  # Convert hex to integer
  start_int = int(start_codepoint, 16)
  end_int = int(end_codepoint, 16)

  # Open the UnicodeData.txt file for reading, and the output file for writing valid codepoints
  with open('/content/UnicodeData.txt' ,'r') as infile, open(os.path.join(scripts_dir,f'{script}.txt'),'w') as outfile:
    for line in infile.readlines():
      # Fields in UnicodeData.txt are separated by semicolons
      parts = line.split(';')

      # Skip malformed or incomplete lines
      if len(parts)<3:
        continue

      # Extract necessary fields
      codepoint, name, category,_, _, decomposition = parts[0], parts[1], parts[2], parts[3], parts[4], parts[5]

      # Skip characters that have a canonical decomposition (i.e., no angle bracket tag like <compat>)
      if decomposition and not decomposition.startswith('<'):
          continue

      # Convert codepoint to integer
      cp_int = int(codepoint, 16)

      # Check if the codepoint lies within the script's range and is in the renderable categories
      if start_int<=cp_int<=end_int:
        if category in renderable_categories:
          outfile.write('U+'+codepoint+'\n')

In [ ]:
count=0
for script in os.listdir(scripts_dir):
  with open(os.path.join(scripts_dir,script),'r') as infile:
    count+=len(infile.readlines())
print(count)

In [ ]:
# Shows the fonts installed in the system
!fc-list

## 3. Download fonts

### 3.1 Download the noto fonts

In [ ]:
!git clone https://github.com/notofonts/notofonts.github.io.git

### 3.2 Download Unifont

In [ ]:
!wget https://unifoundry.com/pub/unifont/unifont-16.0.04/font-builds/unifont-16.0.04.otf -O /content/unifont-16.0.04.otf

### 3.3 Download the Microsoft fonts

In [ ]:
# !git clone https://github.com/pjobson/Microsoft-Fonts.git

### 3.4 Download SIL fonts

In [ ]:
!wget https://software.sil.org/downloads/r/charis/Charis-7.000.zip

In [ ]:
!unzip /content/Charis-7.000.zip -d /content/

In [ ]:
!wget https://software.sil.org/downloads/r/gentium/Gentium-7.000.zip

In [ ]:
!unzip /content/Gentium-7.000.zip -d /content/

In [ ]:
!wget https://software.sil.org/downloads/r/doulos/DoulosSIL-7.000.zip

In [ ]:
!unzip /content/DoulosSIL-7.000.zip -d /content/

In [ ]:
!wget https://software.sil.org/downloads/r/andika/Andika-7.000.zip

In [ ]:
!unzip /content/Andika-7.000.zip -d /content/

In [ ]:
!wget https://software.sil.org/downloads/r/galatia/GalatiaSIL-2.1-web.zip

In [ ]:
!unzip /content/GalatiaSIL-2.1-web.zip -d /content/

### 3.5 Download the Catrinity font

In [ ]:
!wget https://catrinity-font.de/downloads/Catrinity.otf

### 3.6 Transfer the downloaded fonts to the `fonts` folder

In [ ]:
!mkdir -p /content/fonts
# !mv /content/unifont-16.0.04.otf /content/fonts
!mv /content/Charis-7.000/Charis-Regular.ttf /content/fonts

In [ ]:
!mv /content/Gentium-7.000/Gentium-Regular.ttf /content/fonts

In [ ]:
!mv /content/DoulosSIL-7.000/DoulosSIL-Regular.ttf /content/fonts

In [ ]:
!mv /content/Andika-7.000/Andika-Regular.ttf /content/fonts

In [ ]:
!mv /content/GalatiaSIL-2.1-web/GalSILB.ttf /content/fonts
!mv /content/GalatiaSIL-2.1-web/GalSILR.ttf /content/fonts

In [ ]:
!mv /content/Catrinity.otf /content/fonts

### Extract the zip files

In [ ]:
# !rm -rf /content/Microsoft-Fonts

In [ ]:
# import os
# import gzip
# import shutil

# windows_font_dir = '/content/Microsoft-Fonts/2021 - Windows 11/ttf'

# for filename in os.listdir(windows_font_dir):
#     if filename.endswith('.ttf.gz'):
#         gz_path = os.path.join(windows_font_dir, filename)
#         ttf_filename = filename[:-3]  # remove .gz
#         ttf_path = os.path.join(windows_font_dir, ttf_filename)

#         # Extract .ttf.gz to .ttf
#         with gzip.open(gz_path, 'rb') as f_in, open(ttf_path, 'wb') as f_out:
#             shutil.copyfileobj(f_in, f_out)

#         # Rename the extracted .ttf file to a clean name (e.g., remove extra dashes or metadata)
#         base_name = os.path.splitext(ttf_filename)[0]  # remove .ttf
#         clean_name = base_name.split('-')[0].strip() + '.ttf'
#         clean_path = os.path.join(windows_font_dir, clean_name)
#         os.rename(ttf_path, clean_path)

#         # Delete the .ttf.gz file
#         os.remove(gz_path)

# print("Extraction, renaming, and cleanup complete.")

## 4. Create the codepoint to font mapping

In [ ]:
import os

# Take the .ttf and .otf files
root_dirs = ["/content/notofonts.github.io/fonts","/content/fonts","/content/Microsoft-Fonts/2021 - Windows 11/ttf"]
font_paths = []
for root_dir in root_dirs:
  for root, dirs, files in os.walk(root_dir):
      for file in files:
          if file.endswith('.ttf') or file.endswith('.otf'):
              font_paths.append(os.path.join(root, file))

### 4.1 Only keep the regular or basic fonts without any styles like Bold, Italic, etc.

In [ ]:
import os
from collections import defaultdict

# A map from font family name to its candidate font files
font_family_map = defaultdict(list)

def extract_family_name(path):
    filename = os.path.basename(path)
    # Remove style part like -Bold, -Thin, etc.
    name = filename.replace('.ttf', '')
    name = name.split('-')[0]  # e.g., NotoSansOriya-Regular → NotoSansOriya
    return name

# Group fonts by base family name
for path in font_paths:
    family = extract_family_name(path)
    font_family_map[family].append(path)

# Now pick the 'basic' font: prefer Regular.ttf, else first available
pruned_fonts = []

for family, paths in font_family_map.items():
    regular_fonts = [p for p in paths if 'Regular.ttf' in p]
    if regular_fonts:
        pruned_fonts.append(regular_fonts[0])
    else:
        pruned_fonts.append(paths[0])  # fallback

# Optional: sort the result
pruned_fonts.sort()

# Final output
# for font in pruned_fonts:
#     print(font)


In [ ]:
len(font_paths), len(pruned_fonts)

### 4.2 Install the fonts in the system

In [ ]:
import os
import subprocess
from glob import glob

FONT_SOURCE_DIRS = ["/content/notofonts.github.io/fonts","/content/fonts","/content/Microsoft-Fonts/2021 - Windows 11/ttf"]
FONT_INSTALL_DIR = "/usr/share/fonts/truetype/custom/"

os.makedirs(FONT_INSTALL_DIR, exist_ok=True)

# Skip fonts with variable axes like [wght] or [wdth,wght]
def is_variable_font(font_path):
    basename = os.path.basename(font_path)
    return "[" in basename and "]" in basename

# Extensions to include
FONT_EXTENSIONS = ["ttf", "otf"]

# Loop through each font source directory
for src_dir in FONT_SOURCE_DIRS:
    for ext in FONT_EXTENSIONS:
        font_files = glob(os.path.join(src_dir, f"**/*.{ext}"), recursive=True)

        for font_file in font_files:
            if is_variable_font(font_file):
                print(f"Skipping variable font: {font_file}")
                continue

            filename = os.path.basename(font_file)
            installed_path = os.path.join(FONT_INSTALL_DIR, filename)
            subprocess.run(["cp", font_file, installed_path])

# Update font cache
subprocess.run(["fc-cache", "-fv"])

Verify the installation

In [ ]:
!fc-list

In [ ]:
!fc-list | grep -i unifont

In [ ]:
import subprocess

# Run fc-list and decode the output
output = subprocess.check_output(['fc-list'], encoding='utf-8')

# Split the output into lines (each line = one font)
font_list = output.strip().split('\n')
fc_font_names={}

for font in font_list:
  font_name = font.split(':')[1].strip().split(',')[0]
  font_name_striped = ''.join(font_name.split())
  if font_name_striped not in fc_font_names:
    fc_font_names[font_name_striped.lower()]=font_name

# fc_font_names

In [ ]:
fc_font_names['notosans']

'Noto Sans'

### 4.3 Go through all the scripts and create the mapping of the Unicode codepoints to the fonts which can render them.

Reference: https://unix.stackexchange.com/questions/162305/find-the-best-font-for-rendering-a-codepoint

In [ ]:
import subprocess
from tqdm import tqdm

# Directory containing per-script codepoint files (e.g., Latin.txt, Greek.txt)
scripts_dir = 'scripts'

# Get and sort the list of script files
scripts = os.listdir(scripts_dir)
scripts.sort()

# Dictionary to store which fonts support which Unicode codepoints
supported_fonts_mapping={}

# Function to get system-installed fonts that support a given Unicode codepoint
def get_fonts_supporting_codepoint(hex_codepoint):
    # Use 'fc-list' (Fontconfig) with charset filter to list fonts supporting the codepoint
    cmd = ["fc-list", f":charset={hex_codepoint}"]
    result = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)

    # Split and clean the output
    fonts = result.stdout.strip().split('\n') if result.stdout else []
    cleaned = set()
    for line in fonts:
        parts = line.split(':')
        # Extract the font name before any style info
        font_name = parts[1].strip().split(',')[0]
        cleaned.add(font_name)
    return cleaned

# List to store all codepoints across all scripts
all_codepoints = []

# Read all codepoints from script-specific files and collect them
for script in scripts:
  with open(os.path.join(scripts_dir, script), 'r') as infile:
    for line in infile:
      # Line format is 'U+XXXX', extract the hex part after '+'
      codepoint = line.strip().split('+')[1]
      all_codepoints.append(codepoint)

# Analyze each codepoint to find supporting fonts using tqdm for progress display
for codepoint in tqdm(all_codepoints, desc="Analyzing all codepoints"):
  # Get fonts that support this codepoint
  font_names = get_fonts_supporting_codepoint(codepoint)

  # Store or update the font list for this codepoint
  if codepoint not in supported_fonts_mapping:
    supported_fonts_mapping['U+'+codepoint]=font_names
  else:
    supported_fonts_mapping['U+'+codepoint].update(font_names)

In [ ]:
'Noto Sans' in supported_fonts_mapping['U+00A1']

True

In [ ]:
# List of fonts to exclude completely
excluded_fonts = {"humor sans", "comic sans", "comic neue"}  # Add more if needed

supported_fonts_mapping_trimmed = {}

for character, fonts in supported_fonts_mapping.items():
    # Exclude unwanted fonts
    filtered_fonts = [font for font in fonts if font.lower() not in excluded_fonts]

    # Separate into Noto and others
    noto_fonts = [font for font in filtered_fonts if font.startswith('Noto')]
    other_fonts = [font for font in filtered_fonts if not font.startswith('Noto')]

    # Pick up to 2 Noto fonts
    selected_noto_fonts = sorted(noto_fonts)[:2]

    # Combine with other non-Noto fonts
    trimmed_fonts = selected_noto_fonts + other_fonts

    supported_fonts_mapping_trimmed[character] = sorted(trimmed_fonts)


In [ ]:
supported_fonts_mapping_trimmed['U+0021']

In [ ]:
import json

# Save the mapping to a JSON file
json_filename = "supported_fonts_mapping_trimmed.json"

with open(json_filename, 'w', encoding='utf-8') as json_file:
    json.dump(supported_fonts_mapping_trimmed, json_file, indent=4, ensure_ascii=False)

print(f"Supported fonts mapping saved to {json_filename}")

Supported fonts mapping saved to supported_fonts_mapping_trimmed.json


### 4.4 Find the unsupported characters

In [ ]:
unsupported_chars = []

for script in scripts:
    # Extract script name from the filename (e.g., 'Latin.txt' → 'Latin')
    script_name = script.split('.')[0]

    # Open the script file containing codepoints for that script
    with open(os.path.join(scripts_dir, script), 'r', encoding='utf-8') as infile:
        for line in infile:
            # Remove any surrounding whitespace or newline characters
            line = line.strip()
            if not line:
                continue  # skip blank lines

            # If the codepoint is not found in the font support mapping,
            # or if it maps to an empty list (i.e., no fonts support it), consider it unsupported
            if line not in supported_fonts_mapping_trimmed or supported_fonts_mapping_trimmed[line] == []:
                unsupported_chars.append(line)

In [ ]:
len(unsupported_chars)

0

In [ ]:
with open("unsupported_chars_noto_combined.txt",'w', encoding='utf-8') as f:
  f.write(", ".join(unsupported_chars))

### Install Roboto fonts (if required)

In [ ]:
# !git clone https://github.com/googlefonts/roboto-3-classic.git

In [ ]:
# %cd /content/roboto-3-classic

In [ ]:
# !pip install .

In [ ]:
# !pip install -r requirements.txt

In [ ]:
# !sh sources/build.sh

### 4.5 Take at least 5 fonts for each codepoint

In [ ]:
import json

# Set threshold
MAX_FONTS_PER_CODEPOINT = 5

# Trim to at most 5 fonts per codepoint
trimmed_supported_fonts_mapping = {
    cp: fonts[:MAX_FONTS_PER_CODEPOINT]
    for cp, fonts in supported_fonts_mapping_trimmed.items()
}

# Save to JSON file
json_filename = "supported_fonts_mapping_trimmed.json"

with open(json_filename, 'w', encoding='utf-8') as json_file:
    json.dump(trimmed_supported_fonts_mapping, json_file, indent=4, ensure_ascii=False)

print(f"Supported fonts mapping saved to {json_filename}")


Supported fonts mapping saved to supported_fonts_mapping_trimmed.json


In [ ]:
trimmed_supported_fonts_mapping["U+0061"]

['Andika', 'Arial', 'Bahnschrift', 'Calibri', 'Cambria']

In [ ]:
# Sanity check
unsupported_chars = []

for script in scripts:
    script_name = script.split('.')[0]

    with open(os.path.join(scripts_dir, script), 'r', encoding='utf-8') as infile:
        for line in infile:
            line = line.strip()
            if not line:
                continue  # skip blank lines
            if line not in trimmed_supported_fonts_mapping or trimmed_supported_fonts_mapping[line] == []:
                unsupported_chars.append(line)

In [ ]:
len(unsupported_chars)

0

## 5. Generate the dataset

### 5.1 Install the Qahirah Library

In [ ]:
!apt-get -y update
!apt-get -y install libfreetype6 libcairo2 libsm6 libxext6 libfontconfig1 libxrender1 fontconfig libgl1-mesa-glx unzip
!wget https://gitlab.com/ldo/qahirah/-/archive/master/qahirah-master.tar.gz
!tar -xvzf /content/qahirah-master.tar.gz
!mv /content/qahirah-master /content/qahirah
%cd /content/qahirah
!pip install .
%cd ..

### 5.2 Install Python Freetype

In [ ]:
!wget https://gitlab.com/ldo/python_freetype/-/archive/master/python_freetype-master.tar.gz
!tar -xvzf /content/python_freetype-master.tar.gz
!mv /content/python_freetype-master /content/python_freetype
%cd /content/python_freetype
!pip install .
%cd ..

### 5.4 Inherit Custom VisualGenerator Class from the VisualGenerator class.

In [ ]:
from source.vis_gen import VisualGenerator
from tqdm import tqdm

class CustomVisualGenerator(VisualGenerator):
  def generate_dataset_from_json_file(self, file_path, font_styles, antialiases):
    """
      Generates a dataset of rendered images from a JSON file mapping Unicode codepoints to font names.

      Args:
          file_path (str): Path to the JSON file containing a dictionary where keys are codepoints (e.g., "U+0041")
                          and values are lists of font names that support rendering that codepoint.
          font_styles (List[str]): A list of font style names (e.g., ["Regular", "Bold", "Italic"]) to use for rendering.
          antialiases (List[str]): A list of antialiasing options (e.g., ["Default", "None", "Grayscale"]) to apply
                                  during rendering.

      This method processes each codepoint and renders it using each combination of font name, style, and antialiasing
      setting provided. The rendered images are saved to the output directory specified in the class.

      Note:
          - Codepoints that cause errors during rendering are skipped with a warning.
          - This function assumes the fonts are already installed and accessible by name.
          - Output directory will be created if it doesn't exist.
    """
    out_dir_abs = self._get_out_dir_abs_and_check()

    with open(file_path, 'r') as f:
      trimmed_supported_fonts_mapping = json.load(f)

    print(f"Processing {len(trimmed_supported_fonts_mapping)} codepoints...")
    for codepoint, font_names in tqdm(trimmed_supported_fonts_mapping.items(), desc="Codepoints"):
      try:
        code_point = chr(int('0x' + codepoint[2:], 16))
        self.generate_dataset_from_list([code_point], font_names, font_styles, antialiases)
      except Exception as e:
        print(f"Error processing codepoint {codepoint}: {e}")
        continue

    self._check_out_dir = True

  def generate_dataset_from_list(self, code_points, font_names, font_styles, antialiases):
    # Check if out_dir exists and create if not
    out_dir_abs = self._get_out_dir_abs_and_check()

    for font_name in font_names:
      self.font_name = font_name
      for font_style in font_styles:
        self.font_style = font_style
        for antialias in antialiases:
          self.antialias = antialias
          self.visualize_list(code_points)

    # Flag flipped to False in self._get_out_dir_abs_and_check
    self._check_out_dir = True

  def visualize_list(self, code_points, x=None, y=None):
    # Check if out_dir exists and create if not
    out_dir_abs = self._get_out_dir_abs_and_check()

    # Visualize list of code points
    for idx, code_point in enumerate(code_points):
        self.visualize_single(code_point, False, x=x, y=y)

    # Flag flipped to False in self._get_out_dir_abs_and_check
    self._check_out_dir = True

In [ ]:
# Remove the old datasets to make space
!rm -rf data new_data new_data.zip data.zip

### 5.5 Generate the images of the characters

In [ ]:
import multiprocessing

# To generate images faster, multiprocessing is used.
# This function will be called in parallel by multiple processes.
def process_codepoint(args):
  codepoint, font_names, font_styles, antialiases = args
  # Create an instance of your visual generator with a base font (others passed in generate step)
  vg = CustomVisualGenerator(font_name='Noto Sans')
  vg.image_size = 224
  vg.font_size = 210
  # Output directory for generated images
  vg.out_dir = 'data'

  try:
    # Convert codepoint from 'U+XXXX' format to actual Unicode character
    code_point = chr(int('0x' + codepoint[2:], 16))

    # Generate dataset images for this character with different fonts and styles
    vg.generate_dataset_from_list([code_point], font_names, font_styles, antialiases)
  except Exception as e:
    print(f"Error processing codepoint {codepoint}: {e}")

# Path to the JSON file containing codepoint → list of supported fonts mapping
mapping_file = '/content/supported_fonts_mapping_trimmed.json'

# Font styles and anti-aliasing options to be used while rendering
font_styles = ['Bold','Medium','Regular','DemiLight','Light','Thin']
antialiases = ['Default','None']

# Ensure the output directory exists
os.makedirs('data',exist_ok=True)

# Load the font support mapping from file
with open(mapping_file, 'r') as f:
    trimmed_supported_fonts_mapping = json.load(f)

# Prepare the list of inputs to be passed to each worker process
# Each input contains: (codepoint, list of fonts supporting it, font_styles, antialiases)
inputs = [(codepoint, font_names, font_styles, antialiases)
          for codepoint, font_names in trimmed_supported_fonts_mapping.items()]

# Use multiprocessing to parallelize image generation across 4 worker processes
with multiprocessing.Pool(processes=4) as pool:
  for _ in tqdm(pool.imap_unordered(process_codepoint, inputs), total=len(inputs), desc="Rendering"):
    pass

In [ ]:
import os
import shutil

"""
Organizes the dataset in the following structure:

dataset/
├── U+0041/            # Folder for codepoint 'A'
│   ├── img1.png
│   ├── img2.png
│   └── …
├── U+0430/            # Folder for Cyrillic 'а'
│   ├── img1.png
│   ├── img2.png
│   └── …
├── U+0391/            # Folder for Greek 'Α'
│   ├── img1.png
│   ├── img2.png
│   └── ...
 ...

"""

# Directory containing the original images
image_dir = 'data'

# Destination directory where images will be moved into subfolders
dest_dir = 'data_sans_canonical'

# Get a list of image files (with specific extensions) in the source directory
images = [f for f in os.listdir(image_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

for image in images:
  # Extract the codepoint
  folder_name = image.split('_')[0]
  dest_path = os.path.join(dest_dir, folder_name)

  # Create the subfolder if it doesn't already exist
  os.makedirs(dest_path, exist_ok=True)

  # Move the image into the appropriate subfolder
  shutil.move(
      os.path.join(image_dir, image),
      os.path.join(dest_path, image)
  )

In [ ]:
# Make a zip folder
!zip -r data_sans_canonical.zip data_sans_canonical

In [ ]:
vg = CustomVisualGenerator(font_name='Noto Sans')

vg.image_size = 42
vg.font_size = 40
vg.out_dir = 'data'
mapping_file='/content/supported_fonts_mapping_trimmed.json'

vg.generate_dataset_from_json_file(mapping_file, ['Bold','Medium','Regular','DemiLight','Light','Thin'],
                                  ['Default','None'])

## 6. Transfer the data to Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!cp /content/data_sans_canonical.zip /content/drive/MyDrive/Unicode_GSoC_Colab/data